# HW13 — токенизация, инференс BERT-подобной модели, fine-tuning для классификации текста

Учебный pipeline на датасете `emotion` (6 классов эмоций): sanity-check данных, разбор токенизации, инференс **готовой** pretrained-модели (без дообучения на нашем датасете), fine-tuning DistilBERT с выбором лучшего чекпоинта по **validation**, финальная оценка **один раз** на **test**.


In [ ]:
# 2.3.1 Импорты, seed и среда
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import datasets
import sklearn
from datasets import DatasetDict, load_dataset
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("datasets:", datasets.__version__)
print("transformers:", __import__("transformers").__version__)
print("torch:", torch.__version__)
print("sklearn:", sklearn.__version__)
print("Device:", DEVICE)
print("Seed:", SEED)


## 2.3.2 Данные и первичный анализ

Используем датасет `emotion` из библиотеки `datasets`: короткие английские тексты и метка одной из 6 эмоций. Официальные сплиты **train / validation / test** заданы в `datasets`; **validation** и **test** используем полностью. Для обучения на CPU берём **воспроизводимый поднабор train** (4000 примеров, `shuffle` + `seed=SEED`), чтобы ускорить учебный запуск без изменения постановки задачи.


In [ ]:
ds = load_dataset("emotion")
label_names = ds["train"].features["label"].names
train_for_fit = ds["train"].shuffle(seed=SEED).select(range(4000))

print("Классы:", label_names)
print("Размер train (полный датасет):", len(ds["train"]))
print("Размер train (для fine-tuning в этом ноутбуке):", len(train_for_fit))
print("Размер validation:", len(ds["validation"]))
print("Размер test:", len(ds["test"]))

print("\nПримеры (текст → метка), первые 5 из поднабора train:")
for i in range(5):
    row = train_for_fit[i]
    print(f"  [{label_names[row['label']]}] {row['text']}")

print(
    "\nПостановка: многоклассовая классификация коротких фраз по типу эмоции "
    "(sadness, joy, love, anger, fear, surprise)."
)


## 2.3.3 Токенизация

Показываем, как текст превращается во вход трансформера: **токены** (WordPiece), **input_ids**, **attention_mask**, **special tokens** (`[CLS]`, `[SEP]`, `[PAD]`), а также **truncation** и **padding** до фиксированной длины.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Special tokens:")
print("  cls_token / id:", tokenizer.cls_token, tokenizer.cls_token_id)
print("  sep_token / id:", tokenizer.sep_token, tokenizer.sep_token_id)
print("  pad_token / id:", tokenizer.pad_token, tokenizer.pad_token_id)
print("  unk_token / id:", tokenizer.unk_token, tokenizer.unk_token_id)

demo_texts = train_for_fit["text"][:5]
for j, text in enumerate(demo_texts):
    enc = tokenizer(text, truncation=True, padding=False, max_length=128)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    print(f"\n--- Пример {j+1} ---")
    print("TEXT:", text)
    print("TOKENS:", tokens)
    print("input_ids:", enc["input_ids"])
    print("attention_mask:", enc["attention_mask"])

long_fake = "word " * 200
enc_long = tokenizer(
    long_fake,
    truncation=True,
    padding="max_length",
    max_length=32,
    return_tensors="pt",
)
print("\n--- Padding / truncation (max_length=32) ---")
print("Символов во входе:", len(long_fake))
print("Длина последовательности (фикс.):", enc_long["input_ids"].shape[1])
print("Сумма attention_mask (не-pad токены):", int(enc_long["attention_mask"][0].sum()))

tok_lines = []
for t in demo_texts:
    enc = tokenizer(t, truncation=True, padding=False, max_length=128)
    tok_lines.append(
        f"TEXT: {t}\nTOKENS: {tokenizer.convert_ids_to_tokens(enc['input_ids'])}\n"
        f"input_ids: {enc['input_ids']}\nattention_mask: {enc['attention_mask']}\n"
    )
tok_lines.append(
    f"Truncation+padding demo: max_length=32, attention sum={int(enc_long['attention_mask'][0].sum())}\n"
)
(ARTIFACTS / "tokenization_examples.txt").write_text("\n".join(tok_lines), encoding="utf-8")
print("\n(опционально) сохранено:", ARTIFACTS / "tokenization_examples.txt")


## 2.3.4 Инференс готовой pretrained BERT-подобной модели

Запускаем **готовую** модель `distilbert-base-uncased-finetuned-sst-2-english` (бинарная sentiment на SST-2). Пространство меток (POSITIVE/NEGATIVE) не совпадает с 6 классами `emotion`, поэтому это иллюстрация инференса «чужого» head до дообучения под нашу задачу.


In [ ]:
PRETRAINED_INFERENCE_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
infer_device = 0 if torch.cuda.is_available() else -1
clf_pipe = pipeline(
    "text-classification",
    model=PRETRAINED_INFERENCE_MODEL,
    tokenizer=PRETRAINED_INFERENCE_MODEL,
    device=infer_device,
    truncation=True,
)

infer_texts = train_for_fit["text"][:5]
print("Инференс SST-2 на примерах из emotion:\n")
for t in infer_texts:
    out = clf_pipe(t)[0]
    print(f"TEXT: {t}")
    print(f"  -> {out['label']} (score={out['score']:.4f})\n")

print(
    "Вывод: модель оценивает тональность (POSITIVE/NEGATIVE), но не предсказывает "
    "6 эмоций; для emotion нужен fine-tuning с собственной головой классификации."
)


## 2.3.5 Fine-tuning для классификации текста

Одна модель — `distilbert-base-uncased`, задача — `AutoModelForSequenceClassification` на 6 классах. Токенизация в `map` без фиксированного padding (padding в батчах через `DataCollatorWithPadding`). Обучение на поднаборе **train** (4000); выбор лучшего чекпоинта по **macro-F1 на полном validation** — `load_best_model_at_end=True`, `metric_for_best_model=f1_macro`.


In [ ]:
MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 2
LR = 5e-5


def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


tok_tr = train_for_fit.map(tokenize_fn, batched=True, remove_columns=["text"]).rename_column(
    "label", "labels"
)
tok_va = ds["validation"].map(tokenize_fn, batched=True, remove_columns=["text"]).rename_column(
    "label", "labels"
)
tok_te = ds["test"].map(tokenize_fn, batched=True, remove_columns=["text"]).rename_column(
    "label", "labels"
)
tokenized = DatasetDict({"train": tok_tr, "validation": tok_va, "test": tok_te})

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
)
model.to(DEVICE)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


training_args = TrainingArguments(
    output_dir=str(ARTIFACTS / "trainer_output"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    seed=SEED,
    logging_steps=50,
    report_to="none",
    dataloader_pin_memory=False,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Лучшая по validation f1_macro модель загружена (load_best_model_at_end).")
print("best_metric (f1_macro на validation):", trainer.state.best_metric)


## 2.3.6 Оценка качества и краткий анализ ошибок

Считаем **accuracy** и **f1_macro** на **test** (один проход после `trainer.train()`), строим матрицу ошибок, сохраняем примеры в `artifacts/`.


In [ ]:
test_pred = trainer.predict(tokenized["test"])
logits = test_pred.predictions
y_true = test_pred.label_ids
y_pred = np.argmax(logits, axis=-1)

test_accuracy = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"test_accuracy: {test_accuracy:.4f}")
print(f"test_f1_macro: {test_f1_macro:.4f}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    display_labels=label_names,
    xticks_rotation=45,
    ax=ax,
    colorbar=False,
)
plt.tight_layout()
fig.savefig(ARTIFACTS / "confusion_matrix.png", dpi=150)
plt.show()

probs = torch.nn.functional.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
confidences = probs.max(axis=1)

rows = []
for i in range(len(ds["test"])):
    rows.append(
        {
            "text": ds["test"][i]["text"],
            "true_label": label_names[y_true[i]],
            "pred_label": label_names[y_pred[i]],
            "confidence": float(confidences[i]),
        }
    )

wrong = [r for r in rows if r["true_label"] != r["pred_label"]]
correct = [r for r in rows if r["true_label"] == r["pred_label"]]
show = (wrong[:8] + correct)[:10]
pd.DataFrame(show).to_csv(ARTIFACTS / "sample_predictions.csv", index=False)
print("\nПримеры предсказаний (sample_predictions.csv):")
print(pd.DataFrame(show).to_string(index=False))

print("\nНесколько ошибок для анализа:")
for r in wrong[:5]:
    snippet = r["text"] if len(r["text"]) <= 120 else r["text"][:117] + "..."
    print(f"  true={r['true_label']} pred={r['pred_label']} conf={r['confidence']:.3f} | {snippet}")


### Комментарий к ошибкам

На коротких фразах часто путаются близкие по тону классы (например, **joy** и **love**, **sadness** и **fear**). Нейтральные или двусмысленные формулировки дают пограничные случаи с более низкой уверенностью модели. Матрица ошибок показывает, какие пары классов смешиваются чаще всего.
